In [1]:
import polars as pl
import os

In [3]:
DATASET_PATH = "/group/pmc021/amunif/epi-thesis/workflow/15_HepG2 from ENCODE/dataset"

# Loading the BioMart Dataset

In [28]:
col_names = ['gene_id',
                'external_gene_name',
                'chromosome_name',
                'start_position',
                'end_position',
                'strand']

biomart_df = pl.read_csv(os.path.join(DATASET_PATH, "biomart_response.txt"), new_columns=col_names, separator="\t", has_header=False)

In [29]:
biomart_df

gene_id,external_gene_name,chromosome_name,start_position,end_position,strand
str,str,str,i64,i64,i64
"""ENSG00000000003""","""TSPAN6""","""X""",99883667,99894988,-1
"""ENSG00000000005""","""TNMD""","""X""",99839799,99854882,1
"""ENSG00000000419""","""DPM1""","""20""",49551404,49575092,-1
"""ENSG00000000457""","""SCYL3""","""1""",169818772,169863408,-1
"""ENSG00000000460""","""C1orf112""","""1""",169631245,169823221,1
…,…,…,…,…,…
"""ENSG00000259658""","""RP11-89K11.1""","""15""",102277302,102285913,-1
"""ENSG00000259664""","""CTD-2147F2.2""","""15""",97913601,97971182,-1
"""ENSG00000259680""","""RP11-812E19.9""","""16""",33647044,33647696,-1


In [30]:
biomart_df.schema

Schema([('gene_id', String),
        ('external_gene_name', String),
        ('chromosome_name', String),
        ('start_position', Int64),
        ('end_position', Int64),
        ('strand', Int64)])

In [31]:
# Calculate the TSS
biomart_df = biomart_df.with_columns(
    pl.when(pl.col("strand") == 1)
      .then(pl.col("start_position"))
      .otherwise(pl.col("end_position"))
      .alias("tss")
)

In [32]:
biomart_df

gene_id,external_gene_name,chromosome_name,start_position,end_position,strand,tss
str,str,str,i64,i64,i64,i64
"""ENSG00000000003""","""TSPAN6""","""X""",99883667,99894988,-1,99894988
"""ENSG00000000005""","""TNMD""","""X""",99839799,99854882,1,99839799
"""ENSG00000000419""","""DPM1""","""20""",49551404,49575092,-1,49575092
"""ENSG00000000457""","""SCYL3""","""1""",169818772,169863408,-1,169863408
"""ENSG00000000460""","""C1orf112""","""1""",169631245,169823221,1,169631245
…,…,…,…,…,…,…
"""ENSG00000259658""","""RP11-89K11.1""","""15""",102277302,102285913,-1,102285913
"""ENSG00000259664""","""CTD-2147F2.2""","""15""",97913601,97971182,-1,97971182
"""ENSG00000259680""","""RP11-812E19.9""","""16""",33647044,33647696,-1,33647696


In [33]:
# Reformat the chromosome name
biomart_df = biomart_df.with_columns(
    pl.format("chr{}", pl.col("chromosome_name")).alias("chromosome_name")
)

In [34]:
biomart_df

gene_id,external_gene_name,chromosome_name,start_position,end_position,strand,tss
str,str,str,i64,i64,i64,i64
"""ENSG00000000003""","""TSPAN6""","""chrX""",99883667,99894988,-1,99894988
"""ENSG00000000005""","""TNMD""","""chrX""",99839799,99854882,1,99839799
"""ENSG00000000419""","""DPM1""","""chr20""",49551404,49575092,-1,49575092
"""ENSG00000000457""","""SCYL3""","""chr1""",169818772,169863408,-1,169863408
"""ENSG00000000460""","""C1orf112""","""chr1""",169631245,169823221,1,169631245
…,…,…,…,…,…,…
"""ENSG00000259658""","""RP11-89K11.1""","""chr15""",102277302,102285913,-1,102285913
"""ENSG00000259664""","""CTD-2147F2.2""","""chr15""",97913601,97971182,-1,97971182
"""ENSG00000259680""","""RP11-812E19.9""","""chr16""",33647044,33647696,-1,33647696


# Loading the gene expression dataset

In [35]:
# Read the gene expression dataset
gene_exp_df = pl.read_csv(os.path.join(DATASET_PATH, "57epigenomes.RPKM.pc"), separator='\t', truncate_ragged_lines=True)

In [36]:
gene_exp_df

gene_id,E000,E003,E004,E005,E006,E007,E011,E012,E013,E016,E024,E027,E028,E037,E038,E047,E050,E053,E054,E055,E056,E057,E058,E059,E061,E062,E065,E066,E070,E071,E079,E082,E084,E085,E087,E094,E095,E096,E097,E098,E100,E104,E105,E106,E109,E112,E113,E114,E116,E117,E118,E119,E120,E122,E123,E127,E128
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ENSG00000000003""",23.265,43.985,37.413,29.459,21.864,55.649,52.94,71.629,61.292,44.28,63.184,7.49,8.541,0.576,1.393,1.235,5.544,15.933,27.15,6.433,3.812,6.257,10.151,8.898,14.658,0.298,5.605,73.205,20.954,7.645,35.083,6.265,53.039,64.971,9.594,14.46,3.122,13.463,54.677,13.735,2.435,8.833,4.494,36.012,19.252,11.928,5.637,37.989,0.038,42.639,49.983,11.554,11.847,43.723,0.267,13.758,15.818
"""ENSG00000000005""",0.872,1.642,6.498,0.0,0.157,0.003,0.115,0.087,0.055,1.577,0.726,0.0,0.0,0.0,0.0,0.029,0.0,0.051,0.07,0.0,0.0,0.0,0.0,0.0,0.006,0.0,0.0,0.191,0.0,0.018,0.251,0.0,0.566,0.336,0.03,0.0,0.07,0.0,10.67,0.424,0.032,0.524,0.092,0.205,0.134,0.678,0.121,0.0,0.0,0.0,0.0,0.0,0.018,0.0,0.006,0.0,0.0
"""ENSG00000000419""",55.208,35.259,58.308,48.208,37.477,45.923,44.959,40.438,41.97,51.515,35.129,63.304,47.743,45.394,47.041,38.384,37.351,11.078,12.225,27.126,21.572,28.648,44.444,13.179,22.421,28.648,52.753,52.609,15.701,21.769,26.467,7.879,29.927,30.095,32.469,56.167,32.202,26.051,42.731,19.683,67.684,46.172,33.687,39.226,47.562,61.359,54.866,52.215,79.197,107.098,62.811,42.386,54.869,16.652,73.719,56.578,56.371
"""ENSG00000000457""",3.237,2.596,2.345,8.775,2.723,3.7,3.912,5.011,4.158,3.292,3.16,3.683,2.532,7.409,8.577,7.853,17.602,3.295,4.301,3.706,1.523,1.602,3.933,1.877,4.641,5.433,3.417,4.733,3.349,2.222,5.325,2.977,8.497,9.679,5.593,5.731,2.622,3.907,7.649,5.455,10.873,2.529,2.811,6.044,4.526,8.791,5.484,4.829,11.082,8.814,2.646,2.483,2.527,2.549,7.651,4.967,3.714
"""ENSG00000000460""",7.299,6.649,7.838,7.324,0.83,5.354,5.94,5.704,6.213,7.551,7.705,1.04,1.213,3.459,3.813,3.424,6.91,3.855,4.401,2.896,3.246,3.31,6.491,2.782,2.799,2.292,1.137,0.942,4.716,1.12,1.487,1.611,3.408,3.58,0.806,1.25,0.47,1.134,1.69,1.126,0.518,0.717,0.694,1.893,1.952,3.137,1.631,8.001,13.743,25.369,3.373,4.646,2.179,4.099,22.103,3.29,2.491
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259718""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.017,0.0,0.0,0.231,0.0,0.213,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.656,0.0,0.49,0.0,0.0,0.112,0.0,0.0,0.0,0.0,10.274,0.0,0.063,0.0,1.212,0.019,0.0,0.0,2.619,0.819,1.128,16.12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""ENSG00000259741""",17.71,10.639,22.017,19.103,26.684,21.748,23.737,24.791,23.581,18.816,17.317,41.408,33.922,29.116,33.728,39.728,40.943,78.082,82.655,45.977,16.946,14.155,32.761,10.683,26.048,24.214,75.848,12.832,51.611,43.337,38.85,66.718,45.583,38.938,55.274,44.623,39.926,11.838,52.828,15.076,54.477,24.129,26.183,19.587,28.544,12.168,17.54,74.923,22.036,23.527,16.77,25.595,57.26,15.039,8.164,31.558,57.093
"""ENSG00000259752""",0.171,0.032,0.008,0.042,0.015,0.067,0.033,0.078,0.047,0.0,0.09,0.04,0.294,0.175,0.383,0.35,0.421,0.14,0.341,0.055,0.06,0.016,0.014,0.065,0.024,0.325,0.0,0.0,0.09,0.016,0.048,0.195,0.265,0.217,0.026,0.178,0.043,0.224,0.0,0.18,0.004,0.14,0.062,0.086,0.054,0.407,0.043,0.014,0.091,0.028,0.072,0.012,0.046,0.017,0.066,0.04,0.0


In [37]:
# Join with valid gene from biomart
filtered_gene_exp_df = gene_exp_df.join(
    biomart_df,
    on = "gene_id",
    how = "semi"
)

In [38]:
filtered_gene_exp_df

gene_id,E000,E003,E004,E005,E006,E007,E011,E012,E013,E016,E024,E027,E028,E037,E038,E047,E050,E053,E054,E055,E056,E057,E058,E059,E061,E062,E065,E066,E070,E071,E079,E082,E084,E085,E087,E094,E095,E096,E097,E098,E100,E104,E105,E106,E109,E112,E113,E114,E116,E117,E118,E119,E120,E122,E123,E127,E128
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ENSG00000000003""",23.265,43.985,37.413,29.459,21.864,55.649,52.94,71.629,61.292,44.28,63.184,7.49,8.541,0.576,1.393,1.235,5.544,15.933,27.15,6.433,3.812,6.257,10.151,8.898,14.658,0.298,5.605,73.205,20.954,7.645,35.083,6.265,53.039,64.971,9.594,14.46,3.122,13.463,54.677,13.735,2.435,8.833,4.494,36.012,19.252,11.928,5.637,37.989,0.038,42.639,49.983,11.554,11.847,43.723,0.267,13.758,15.818
"""ENSG00000000005""",0.872,1.642,6.498,0.0,0.157,0.003,0.115,0.087,0.055,1.577,0.726,0.0,0.0,0.0,0.0,0.029,0.0,0.051,0.07,0.0,0.0,0.0,0.0,0.0,0.006,0.0,0.0,0.191,0.0,0.018,0.251,0.0,0.566,0.336,0.03,0.0,0.07,0.0,10.67,0.424,0.032,0.524,0.092,0.205,0.134,0.678,0.121,0.0,0.0,0.0,0.0,0.0,0.018,0.0,0.006,0.0,0.0
"""ENSG00000000419""",55.208,35.259,58.308,48.208,37.477,45.923,44.959,40.438,41.97,51.515,35.129,63.304,47.743,45.394,47.041,38.384,37.351,11.078,12.225,27.126,21.572,28.648,44.444,13.179,22.421,28.648,52.753,52.609,15.701,21.769,26.467,7.879,29.927,30.095,32.469,56.167,32.202,26.051,42.731,19.683,67.684,46.172,33.687,39.226,47.562,61.359,54.866,52.215,79.197,107.098,62.811,42.386,54.869,16.652,73.719,56.578,56.371
"""ENSG00000000457""",3.237,2.596,2.345,8.775,2.723,3.7,3.912,5.011,4.158,3.292,3.16,3.683,2.532,7.409,8.577,7.853,17.602,3.295,4.301,3.706,1.523,1.602,3.933,1.877,4.641,5.433,3.417,4.733,3.349,2.222,5.325,2.977,8.497,9.679,5.593,5.731,2.622,3.907,7.649,5.455,10.873,2.529,2.811,6.044,4.526,8.791,5.484,4.829,11.082,8.814,2.646,2.483,2.527,2.549,7.651,4.967,3.714
"""ENSG00000000460""",7.299,6.649,7.838,7.324,0.83,5.354,5.94,5.704,6.213,7.551,7.705,1.04,1.213,3.459,3.813,3.424,6.91,3.855,4.401,2.896,3.246,3.31,6.491,2.782,2.799,2.292,1.137,0.942,4.716,1.12,1.487,1.611,3.408,3.58,0.806,1.25,0.47,1.134,1.69,1.126,0.518,0.717,0.694,1.893,1.952,3.137,1.631,8.001,13.743,25.369,3.373,4.646,2.179,4.099,22.103,3.29,2.491
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""",1.093,1.31,0.335,0.942,2.077,0.465,0.631,0.806,1.458,0.708,0.425,0.801,0.107,3.197,6.251,5.575,12.799,0.927,0.682,1.654,0.704,0.444,0.339,0.398,1.032,3.029,0.34,0.212,0.999,1.388,1.479,0.415,1.563,0.794,0.8,0.5,0.461,3.337,4.112,0.678,0.093,0.929,0.883,1.5,0.792,14.418,5.263,1.178,11.668,0.0,1.842,0.048,0.491,2.024,3.524,0.19,0.824
"""ENSG00000259664""",0.143,0.004,0.0,0.0,0.003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.041,0.015,0.156,0.003,0.006,0.012,0.0,0.267,0.026,0.0,0.0,0.045,0.005,0.288,0.0,0.0,0.094,0.0,0.0,0.012,0.0,0.0,0.0,0.0,0.0,0.004,0.0,0.0
"""ENSG00000259680""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.226,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.224,0.0,0.071,0.0,0.0,0.037,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.017,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [39]:
# Only select the E118 (HepG2)
E118_df = filtered_gene_exp_df.select(["gene_id", "E118"])

In [40]:
E118_df

gene_id,E118
str,f64
"""ENSG00000000003""",49.983
"""ENSG00000000005""",0.0
"""ENSG00000000419""",62.811
"""ENSG00000000457""",2.646
"""ENSG00000000460""",3.373
…,…
"""ENSG00000259658""",1.842
"""ENSG00000259664""",0.0
"""ENSG00000259680""",0.0


# Join with Biomart data

In [41]:
E118_merged_df = E118_df.join(biomart_df, on="gene_id")

In [42]:
E118_merged_df

gene_id,E118,external_gene_name,chromosome_name,start_position,end_position,strand,tss
str,f64,str,str,i64,i64,i64,i64
"""ENSG00000000003""",49.983,"""TSPAN6""","""chrX""",99883667,99894988,-1,99894988
"""ENSG00000000005""",0.0,"""TNMD""","""chrX""",99839799,99854882,1,99839799
"""ENSG00000000419""",62.811,"""DPM1""","""chr20""",49551404,49575092,-1,49575092
"""ENSG00000000457""",2.646,"""SCYL3""","""chr1""",169818772,169863408,-1,169863408
"""ENSG00000000460""",3.373,"""C1orf112""","""chr1""",169631245,169823221,1,169631245
…,…,…,…,…,…,…,…
"""ENSG00000259658""",1.842,"""RP11-89K11.1""","""chr15""",102277302,102285913,-1,102285913
"""ENSG00000259664""",0.0,"""CTD-2147F2.2""","""chr15""",97913601,97971182,-1,97971182
"""ENSG00000259680""",0.0,"""RP11-812E19.9""","""chr16""",33647044,33647696,-1,33647696


# Preparing dataset for bedtools intersect

In [46]:
# Find the +/- 5,000 bp location from TSS
# Make sure that the start is smaller than the end

E118_merged_df = E118_merged_df.with_columns(
    (pl.col("tss") - 5000).alias("start"),
    (pl.col("tss") + 5000).alias("end")
)

In [47]:
E118_merged_df

gene_id,E118,external_gene_name,chromosome_name,start_position,end_position,strand,tss,start,end
str,f64,str,str,i64,i64,i64,i64,i64,i64
"""ENSG00000000003""",49.983,"""TSPAN6""","""chrX""",99883667,99894988,-1,99894988,99889988,99899988
"""ENSG00000000005""",0.0,"""TNMD""","""chrX""",99839799,99854882,1,99839799,99834799,99844799
"""ENSG00000000419""",62.811,"""DPM1""","""chr20""",49551404,49575092,-1,49575092,49570092,49580092
"""ENSG00000000457""",2.646,"""SCYL3""","""chr1""",169818772,169863408,-1,169863408,169858408,169868408
"""ENSG00000000460""",3.373,"""C1orf112""","""chr1""",169631245,169823221,1,169631245,169626245,169636245
…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""",1.842,"""RP11-89K11.1""","""chr15""",102277302,102285913,-1,102285913,102280913,102290913
"""ENSG00000259664""",0.0,"""CTD-2147F2.2""","""chr15""",97913601,97971182,-1,97971182,97966182,97976182
"""ENSG00000259680""",0.0,"""RP11-812E19.9""","""chr16""",33647044,33647696,-1,33647696,33642696,33652696


In [48]:
E118_merged_df.schema

Schema([('gene_id', String),
        ('E118', Float64),
        ('external_gene_name', String),
        ('chromosome_name', String),
        ('start_position', Int64),
        ('end_position', Int64),
        ('strand', Int64),
        ('tss', Int64),
        ('start', Int64),
        ('end', Int64)])

In [49]:
# Reorder the column
E118_merged_df = E118_merged_df.select([
    "chromosome_name",
    "start",
    "end",
    "gene_id",
    "E118",
    "strand",
    "external_gene_name",
    "start_position",
    "end_position",
    "tss"
])

In [50]:
E118_merged_df

chromosome_name,start,end,gene_id,E118,strand,external_gene_name,start_position,end_position,tss
str,i64,i64,str,f64,i64,str,i64,i64,i64
"""chrX""",99889988,99899988,"""ENSG00000000003""",49.983,-1,"""TSPAN6""",99883667,99894988,99894988
"""chrX""",99834799,99844799,"""ENSG00000000005""",0.0,1,"""TNMD""",99839799,99854882,99839799
"""chr20""",49570092,49580092,"""ENSG00000000419""",62.811,-1,"""DPM1""",49551404,49575092,49575092
"""chr1""",169858408,169868408,"""ENSG00000000457""",2.646,-1,"""SCYL3""",169818772,169863408,169863408
"""chr1""",169626245,169636245,"""ENSG00000000460""",3.373,1,"""C1orf112""",169631245,169823221,169631245
…,…,…,…,…,…,…,…,…,…
"""chr15""",102280913,102290913,"""ENSG00000259658""",1.842,-1,"""RP11-89K11.1""",102277302,102285913,102285913
"""chr15""",97966182,97976182,"""ENSG00000259664""",0.0,-1,"""CTD-2147F2.2""",97913601,97971182,97971182
"""chr16""",33642696,33652696,"""ENSG00000259680""",0.0,-1,"""RP11-812E19.9""",33647044,33647696,33647696


In [52]:
E118_merged_df.write_csv(
    os.path.join(DATASET_PATH, "E118.bed"),
    separator = "\t",
    include_header = False
)